# 10 — V6.2E parent prepare (low disk)
**V6.2E is immutable** (`v6p2e_23562a45b343`, 17,483 / 2,272 / 2,012).
1. download freeze + split manifest, validate against config (conflicts stop the run);
2. index base/overlay archives without extraction;
3. stream ≤1200 TRAIN backgrounds with an insulator anchor;
4. pHash VAL+TEST once → `parent_val_test_phash.csv.gz` (incremental checkpoint);
5. close handles and delete the large archives (`runtime.clean_parent_archives_after_qa`).
Re-running resumes; a complete matching result is reused.

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
pipeline.run_stage("parent", dry_run=True)

In [ ]:
outcome = pipeline.run_stage("parent", force=FORCE_REBUILD, dry_run=DRY_RUN)
print(outcome)
display(pd.Series(outcome.summary, dtype=object).to_frame("value"))

In [ ]:
pool = pd.read_csv(paths.resolve(store.load("parent").output("bg_pool_csv")["path"]))
display(pool.head())
for p in pool["image"].head(3):
    display(IPImage(filename=str(paths.resolve(p)), width=480))